# 06 — Sample Ratio Mismatch (SRM)
**Prerequisites:** `02_randomization_methods.ipynb` (hash-based bucketing, balance checks).
**Connects to:** `01_ab_testing_fundamentals.ipynb` (pipeline validation step); `07_multiple_testing.ipynb`.

## Narrative thread
```
SRM definition -> chi-square goodness-of-fit test -> common causes
   -> worked detection example -> debugging decision tree
```

## What is SRM?

**Sample Ratio Mismatch** occurs when the *observed* allocation of users across experiment arms
differs from the *intended* allocation (e.g., you configured a 50/50 split but observe 50.6% vs
49.4% at a sample size where that gap is not plausible noise). SRM is one of the most important
things to check before trusting *any* other result from an experiment: if the arms aren't the
populations you think they are, the metric comparison is comparing two different underlying
populations, not treatment vs. control on the same population — as Fabijan et al. and Kohavi et
al. both emphasize, it is a leading cause of **wrong ship decisions** in practice, and is entirely
independent of whatever the primary metric says.

## The chi-square goodness-of-fit test

Given observed counts $n_1, \dots, n_k$ across $k$ arms and intended allocation proportions
$p_1, \dots, p_k$ (e.g. $p_1=p_2=0.5$), the test statistic is

$$\chi^2 = \sum_{i=1}^{k} \frac{(n_i - n p_i)^2}{n p_i} \sim \chi^2_{k-1} \text{ under } H_0$$

Because online experiments often have very large $n$, this test is extremely sensitive: even a
tiny, practically irrelevant deviation (e.g. 50.05% vs 49.95%) can produce a significant p-value.
Practitioners commonly use a strict threshold like $p < 0.001$ (rather than 0.05) specifically
*because* of this high sensitivity at scale — a routine 0.05 threshold would flag SRM constantly
on pure noise.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [2]:
# ── Worked SRM detection example ─────────────────────────────────────────
def srm_check(counts, expected_props, alpha=0.001):
    counts = np.asarray(counts, dtype=float)
    n = counts.sum()
    expected = n * np.asarray(expected_props)
    chi2, p_value = stats.chisquare(counts, expected)
    flagged = p_value < alpha
    return chi2, p_value, flagged

# Case 1: healthy experiment, intended 50/50, tiny natural fluctuation
np.random.seed(5)
n_total = 2_000_000
healthy_counts = [n_total // 2 + 150, n_total // 2 - 150]
chi2, p, flag = srm_check(healthy_counts, [0.5, 0.5])
print(f"Healthy: counts={healthy_counts}, chi2={chi2:.3f}, p={p:.4f}, SRM flagged (p<0.001)? {flag}")

# Case 2: a bucketing/redirect bug causes a real, business-meaningful imbalance
buggy_counts = [n_total // 2 + 8000, n_total // 2 - 8000]  # 0.8pp skew: control gets systematically more traffic
chi2, p, flag = srm_check(buggy_counts, [0.5, 0.5])
print(f"Buggy:   counts={buggy_counts}, chi2={chi2:.3f}, p={p:.2e}, SRM flagged (p<0.001)? {flag}")

print("\nNote how a mere 0.8 percentage-point deviation from 50/50 is already astronomically ")
print("significant at n=2,000,000 — this is exactly why SRM checks use a stringent alpha,")
print("and why a 'statistically significant' SRM at huge n still needs a judgment call on")
print("whether the *magnitude* is large enough to invalidate the read.")

Healthy: counts=[1000150, 999850], chi2=0.045, p=0.8320, SRM flagged (p<0.001)? False
Buggy:   counts=[1008000, 992000], chi2=128.000, p=1.12e-29, SRM flagged (p<0.001)? True

Note how a mere 0.8 percentage-point deviation from 50/50 is already astronomically 
significant at n=2,000,000 — this is exactly why SRM checks use a stringent alpha,
and why a 'statistically significant' SRM at huge n still needs a judgment call on
whether the *magnitude* is large enough to invalidate the read.


## Common causes of SRM

| Cause | Mechanism |
|---|---|
| **Bucketing bugs** | Hash function bug, off-by-one in bucket boundaries, salt collision — see `02_randomization_methods.ipynb` |
| **Bot / crawler filtering applied asymmetrically** | If bot-filtering logic runs *after* assignment but interacts with the treatment code path differently (e.g. treatment redirects trigger bot detectors more often), one arm loses more "users" post hoc |
| **Redirect / latency bugs** | If treatment requires an extra network round-trip and some users time out or bounce before the exposure event logs, treatment's logged population shrinks non-randomly |
| **Client-side logging failures** | A JS/analytics bug that only fires (or fails to fire) in one arm's UI drops exposure events for that arm |
| **Residual/leftover traffic from a prior experiment** | Reusing an experiment ID/salt without fully resetting state |

## Debugging decision tree

```
SRM flagged (chi-square p < threshold)
│
├── Is the skew large enough to matter practically (not just statistically)?
│     ├── No (huge n, trivial % gap) -> monitor, but proceed with caution
│     └── Yes -> continue
│
├── Check assignment layer: are raw bucket counts (pre-exposure-logging) balanced?
│     ├── Imbalanced at the RANDOMIZER -> bucketing/hash/salt bug (fix in 02_randomization_methods.ipynb layer)
│     └── Balanced at the randomizer -> continue
│
├── Check exposure logging: do assignment counts == exposure-event counts, per arm?
│     ├── Divergence only in one arm -> client logging / redirect / latency bug in that arm's code path
│     └── No divergence -> continue
│
├── Check post-hoc filters: bot detection, fraud filters, outlier trimming
│     ├── Filters remove different fractions per arm -> filtering logic bug or genuine
│     │   arm-dependent bot behavior (e.g. treatment triggers more/less automated traffic)
│     └── Filters symmetric -> re-examine infra logs, CDN routing, feature-flag rollout %
│
└── If no root cause found: DO NOT trust the experiment's metric results. Re-run after a fix.
```

## Key takeaways

| Concept | Statement |
|---|---|
| SRM | Observed arm allocation differs from intended allocation — invalidates the comparison |
| Detection | Chi-square goodness-of-fit test, typically with a strict threshold (e.g. p < 0.001) due to high sensitivity at scale |
| Causes | Bucketing bugs, asymmetric bot filtering, redirect/logging bugs, salt reuse |
| Response | Root-cause and fix before trusting any metric from the same run — never "explain away" SRM statistically |

## References

- Xie, H., & Aurisset, J. (2016). Diagnosing Sample Ratio Mismatch in Online Controlled Experiments. *KDD 2016*.
- Fabijan, A., Dmitriev, P., Olsson, H. H., & Bosch, J. (2017). The Benefits of Controlled Experimentation at Scale. *EUROMICRO SEAA*.
- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*, Ch. 3 & 21 (SRM as a top trust-eroding bug class).
